In [2]:
import pandas as pd
import re
from pathlib import Path

# ==================================================
# Duration Parser
# ==================================================

folder_path = Path(
    r"C:\Users\mturky\Documents\cms reports"
)

def parse_duration(value):

    if isinstance(value, pd.Timedelta):
        return value

    if pd.isna(value):
        return pd.Timedelta(seconds=0)

    value = str(value).strip()

    if value == "" or value.lower() == "nan":
        return pd.Timedelta(seconds=0)

    if "day" in value:
        return pd.to_timedelta(value)

    # :20 => 20 sec
    if value.startswith(":") and value.count(":") == 1:
        return pd.Timedelta(seconds=int(value[1:]))

    # :12:00 => 12 min 0 sec
    if value.startswith(":") and value.count(":") == 2:
        mins, secs = value[1:].split(":")
        return pd.Timedelta(
            minutes=int(mins),
            seconds=int(secs)
        )

    parts = value.split(":")

    # 24:39:00 => 24 min 39 sec
    if len(parts) == 3:
        return pd.Timedelta(
            minutes=int(parts[0]),
            seconds=int(parts[1])
        )

    # 5:04 => 5 min 4 sec
    if len(parts) == 2:
        return pd.Timedelta(
            minutes=int(parts[0]),
            seconds=int(parts[1])
        )

    return pd.Timedelta(seconds=0)


def get_shift(t):

    if pd.isna(t):
        return None

    hour = t.hour
    shifts = []

    if 8 <= hour < 16:
        shifts.append("08:00 AM - 04:00 PM")

    if 12 <= hour < 20:
        shifts.append("12:00 PM - 08:00 PM")

    if 16 <= hour or hour == 0:
        shifts.append("04:00 PM - 12:00 AM")

    if hour >= 20 or hour < 8:
        shifts.append("08:00 PM - 08:00 AM")

    return ", ".join(shifts)

# ==================================================
# Process One HTML File
# ==================================================

def process_html_file(file):

    # ----------------------------------------------
    # Read HTML
    # ----------------------------------------------

    with open(file, "r", encoding="utf-8", errors="ignore") as f:
        html = f.read()

    # ----------------------------------------------
    # Extract Report Date
    # ----------------------------------------------

    match = re.search(
    r"Date:.*?<B>(.*?)</B>",
    html,
    re.IGNORECASE | re.DOTALL
)

    report_date = match.group(1).strip() if match else None

    if report_date:

        report_date = report_date.replace("-", "-").strip()

        parsed_date = None

        for fmt in (
            "%d/%m/%y",     # 18/06/26
            "%d/%m/%Y",     # 18/06/2026
            "%d-%b-%y",     # 10-Jun-26
            "%d-%b-%Y"      # 10-Jun-2026
        ):

            try:
                parsed_date = pd.to_datetime(
                    report_date,
                    format=fmt
                )
                break

            except ValueError:
                pass

        if parsed_date is not None:
            report_date = parsed_date.strftime("%d/%m/%Y")
        else:
            report_date = None

    # ----------------------------------------------
    # Read Tables
    # ----------------------------------------------

    tables = pd.read_html(file)

    if not tables:
        return None, None

    df = tables[0].copy()

    # ----------------------------------------------
    # Extract Totals Row
    # ----------------------------------------------

    totals_df = pd.DataFrame()

    if "Time" in df.columns:

        totals_df = (
            df[df["Time"] == "Totals"]
            .copy()
            .reset_index(drop=True)
        )

        df = (
            df[df["Time"] != "Totals"]
            .copy()
            .reset_index(drop=True)
        )

    duration_cols = [
        "Avg Speed Ans",
        "Avg Aban Time",
        "Avg ACD Time",
        "Avg ACW Time",
        "AUX Time",
        "Max Delay"
    ]

    for col in duration_cols:
        if col in totals_df.columns:
            totals_df[col] = totals_df[col].apply(parse_duration)

            totals_df[f"{col} seconds"] = (
                totals_df[col]
                .dt.total_seconds()
            )

    if not totals_df.empty:
        totals_df["Report Date"] = report_date

    # ----------------------------------------------
    # Rename Third Column To "To"
    # ----------------------------------------------

    if len(df.columns) > 2:
        cols = list(df.columns)
        cols[2] = "To"
        df.columns = cols

    # ----------------------------------------------
    # Remove Separator Column
    # ----------------------------------------------

    if len(df.columns) > 1:
        df = df.drop(df.columns[1], axis=1)

    # ----------------------------------------------
    # Remove Unneeded Columns
    # ----------------------------------------------

    df = df.drop(
        columns=["Flow In", "Flow Out"],
        errors="ignore"
    )

    df = df.reset_index(drop=True)

    # ----------------------------------------------
    # Numeric Columns
    # ----------------------------------------------

    numeric_cols = [
        "ACD Calls",
        "Aban Calls",
        "AVG STAFF"
    ]

    for col in numeric_cols:

        if col in df.columns:

            df[col] = (
                pd.to_numeric(
                    df[col],
                    errors="coerce"
                )
                .fillna(0)
            )

    # ----------------------------------------------
    # Duration Columns
    # ----------------------------------------------

    for col in duration_cols:

        if col in df.columns:

            df[col] = df[col].apply(parse_duration)

            df[f"{col} seconds"] = (
                df[col]
                .dt.total_seconds()
            )

    # ----------------------------------------------
    # Time Columns
    # ----------------------------------------------

    if "To" in df.columns:

        original_to = df["To"].astype(str).str.strip()

        # Try AM/PM format first
        df["To"] = pd.to_datetime(
            original_to,
            format="%I:%M%p",
            errors="coerce"
        )

        # Try 24-hour format for failed rows
        mask = df["To"].isna()

        df.loc[mask, "To"] = pd.to_datetime(
            original_to[mask],
            format="%H:%M",
            errors="coerce"
        )

        df["Time"] = (
            df["To"]
            - pd.Timedelta(minutes=30)
        )

        df["Shift"] = df["Time"].apply(get_shift)

        df["To"] = df["To"].dt.strftime("%I:%M %p")
        df["Time"] = df["Time"].dt.strftime("%I:%M %p")

    # ----------------------------------------------
    # Add Report Date
    # ----------------------------------------------

    df["Report Date"] = report_date

    return df, totals_df


# ==================================================
# Folder Processing
# ==================================================



all_details = []
all_totals = []

html_files = list(folder_path.glob("*.HTML"))

print(f"Found {len(html_files)} HTML files")

for file in html_files:

    # print(f"Processing {file.name}")

    try:

        detail_df, totals_df = process_html_file(file)

        if detail_df is not None and not detail_df.empty:
            all_details.append(detail_df)

        if totals_df is not None and not totals_df.empty:
            all_totals.append(totals_df)

    except Exception as e:

        print(
            f"Error processing {file.name}: {e}"
        )

# ==================================================
# Create Final DataFrames
# ==================================================

if all_details:

    final_df = pd.concat(
        all_details,
        ignore_index=True
    )

    final_df.to_csv(
        "bcms_all_reports.csv",
        index=False
    )

    print(
        f"Saved bcms_all_reports.csv "
        f"({len(final_df):,} rows)"
    )

else:

    final_df = pd.DataFrame()

if all_totals:

    final_totals_df = pd.concat(
        all_totals,
        ignore_index=True
    )

    final_totals_df.to_csv(
        "bcms_all_totals.csv",
        index=False
    )

    print(
        f"Saved bcms_all_totals.csv "
        f"({len(final_totals_df):,} rows)"
    )

else:

    final_totals_df = pd.DataFrame()

# ==================================================
# Summary
# ==================================================

print("\nSummary")
print("-" * 40)
print(f"Files Processed : {len(html_files)}")
print(f"Detail Rows     : {len(final_df):,}")
print(f"Totals Rows     : {len(final_totals_df):,}")

print("\nFiles Created:")
print("bcms_all_reports.csv")
print("bcms_all_totals.csv")

Found 8 HTML files
Saved bcms_all_reports.csv (10,752 rows)
Saved bcms_all_totals.csv (8 rows)

Summary
----------------------------------------
Files Processed : 8
Detail Rows     : 10,752
Totals Rows     : 8

Files Created:
bcms_all_reports.csv
bcms_all_totals.csv
